DELTA LAKE CONCEPTS - DATABRICKS NOTEBOOK

Topics: Schema Enforcement & Evolution | Time Travel | Cloning | Partitioning & Z-Ordering

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, lit
from datetime import date

# Define schema explicitly
schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("city",       StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("join_date",  DateType(),    True),
])

# Sample data
data = [
    (101, "Arjun",    "Engineering", "Chennai",   85000.0, date(2020, 3, 15)),
    (102, "Priya",    "Marketing",   "Mumbai",    72000.0, date(2019, 7, 1)),
    (103, "Karthik",  "Engineering", "Bangalore", 91000.0, date(2021, 1, 10)),
    (104, "Sneha",    "HR",          "Chennai",   65000.0, date(2018, 11, 20)),
    (105, "Rahul",    "Finance",     "Delhi",     78000.0, date(2022, 5, 5)),
    (106, "Divya",    "Engineering", "Bangalore", 88000.0, date(2020, 9, 3)),
    (107, "Manoj",    "Marketing",   "Mumbai",    70000.0, date(2023, 2, 14)),
    (108, "Anitha",   "HR",          "Chennai",   63000.0, date(2017, 8, 25)),
]

df = spark.createDataFrame(data, schema)
df.show()

target = "employeeproject.default.employee"
# Also register as a SQL table for SQL-style queries
spark.sql(f"""DROP TABLE IF EXISTS {target}""")
df.write.mode("overwrite").saveAsTable(f"""{target}""")

print("Sample Delta table created successfully!")



CONCEPT 1: SCHEMA ENFORCEMENT & EVOLUTION


In [0]:
# -----------------------------------------------------------------------------
# 1A. Schema Enforcement — Delta rejects writes with mismatched schema
# -----------------------------------------------------------------------------

print("\n--- 1A. Schema Enforcement ---")

# Bad data: extra column 'bonus' not in original schema
bad_data = [(109, "Vikram", "Sales", "Pune", 74000.0, date(2023, 6, 1), 5000.0)]
bad_schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("city",       StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("join_date",  DateType(),    True),
    StructField("bonus",      DoubleType(),  True),   # 🆕 Extra column
])

bad_df = spark.createDataFrame(bad_data, bad_schema)

try:
    bad_df.write.mode("append").saveAsTable(f"""{target}""")
except Exception as e:
    print(f"❌ Schema Enforcement Error (Expected): {e}")
    print("✅ Delta blocked the write because 'bonus' column doesn't exist in the table schema.")



In [0]:
# -----------------------------------------------------------------------------
# 1B. Schema Evolution — Allow new columns using mergeSchema
# -----------------------------------------------------------------------------

print("\n--- 1B. Schema Evolution (mergeSchema) ---")

bad_df.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"""{target}""")

evolved_df = spark.read.table(f"""{target}""")
evolved_df.show()
print("Schema evolved: 'bonus' column added. Existing rows show NULL for the new column.")



Time Travel

In [0]:
# -----------------------------------------------------------------------------
# 2A. Check table history
# -----------------------------------------------------------------------------

from delta.tables import DeltaTable

target = "employeeproject.default.employee"

delta_table = DeltaTable.forName(spark, target)

display(
    delta_table.history().select("version", "timestamp", "operation")
)

# -----------------------------------------------------------------------------
# 2B. Query previous version by VERSION number
# -----------------------------------------------------------------------------

print(" Reading Version 0 (original data — before schema evolution):")
df_v0 = spark.read \
    .option("versionAsOf", 0) \
    .table(target)
display(df_v0)


# -----------------------------------------------------------------------------
# 2C. Query previous version by TIMESTAMP
# -----------------------------------------------------------------------------

from pyspark.sql.functions import col

# Get the timestamp of version 0 from history
v0_timestamp = delta_table.history() \
    .filter(col("version") == 0) \
    .select("timestamp") \
    .collect()[0][0]

print(f"Reading data as of timestamp: {v0_timestamp}")

df_ts = spark.read \
    .format("delta") \
    .option("timestampAsOf", str(v0_timestamp)) \
    .table(target)
df_ts.show()


# -----------------------------------------------------------------------------
# 2D. Restore table to a previous version (RESTORE command)
# -----------------------------------------------------------------------------

print("Restoring table to Version 0:")
spark.sql(f"""RESTORE TABLE {target} TO VERSION AS OF 0""")
spark.sql(f"""SELECT * FROM {target}""").show()
print("✅ Table restored to original state (Version 0).")




In [0]:
# =============================================================================
# CONCEPT 4: PARTITIONING & Z-ORDERING
# =============================================================================

print("\n--- 4A. Partitioning ---")

# -----------------------------------------------------------------------------
# 4A. Write Delta table with Partitioning
# Choose low-cardinality column (department has ~4 values — good for partitioning)
# -----------------------------------------------------------------------------

target_new = "employeeproject.default.employee_partitioned"

df_v0.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("department") \
    .saveAsTable(f"{target_new}")

print("✅ Delta table written with partitioning on 'department'.")
print("   Folder structure: /employee_partitioned/department=Engineering/")
print("                      /employee_partitioned/department=HR/")
print("                      /employee_partitioned/department=Marketing/ ...")

# Query with partition pruning — only Engineering files are scanned
partitioned_df = spark.read \
    .format("delta") \
    .table(f"{target_new}") \
    .filter(col("department") == "Engineering")

partitioned_df.show()
print("✅ Partition pruning: Only 'Engineering' partition files were scanned.")


# -----------------------------------------------------------------------------
# 4B. Z-Ordering — co-locate related data within each file (for high-cardinality columns)
# -----------------------------------------------------------------------------

print("\n--- 4B. Z-Ordering ---")

# OPTIMIZE + ZORDER — co-locates rows by 'city' within each partition file
# Best for columns used frequently in filters/joins but have too many unique values for partitioning
spark.sql(f"""
    OPTIMIZE {target_new}
    ZORDER BY (city)
""")
print("   Z-Ordering applied on 'city' column.")
print("   Rows with same city are now co-located within Parquet files.")
print("   Queries filtering by city will scan far fewer row groups.")

print("- **********Partition column → visible in the path as subdirectories.")
print("- **********Z-Order column → invisible in the path, but improves file-level data skipping and query performance")


# -----------------------------------------------------------------------------
# 4D. Bonus — Check table statistics after OPTIMIZE
# -----------------------------------------------------------------------------

spark.sql(f"DESCRIBE DETAIL {target_new}").select(
    "name", "numFiles", "sizeInBytes"
).show(truncate=False)

In [0]:
%sql
describe detail employeeproject.default.employee_partitioned

In [0]:
%fs ls /user/hive/warehouse/employeeproject/default/employee_partitioned/department=Engineering/